# 記事の見出しを生成して実際の見出しと比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_summarization/headline_generation.ipynb)

NLP クックブック 24-10 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

記事の本文から、内容を一言で表す見出し（タイトル）を作りたい。極端に短い要約として、生成結果を実際の見出しと比べたい。

In [1]:
!pip install -q rouge-score fugashi unidic-lite

In [2]:
import io, tarfile, urllib.request
import pandas as pd
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
df = df[df["label"].isin(["topic-news", "sports-watch"])].sample(12, random_state=1).reset_index(drop=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる (LLM の章の「オープンな LLM とチャットする」を参照)。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def headline(text):
    prompt = ("次のニュース記事に、30 字以内の見出しを 1 つ付けてください。体言止めにし、記事にない情報は入れないでください。見出しだけを出力してください。\n\n" + text[:600])
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0].strip("「」『』")

df["generated"] = [headline(t) for t in df["text"]]   # T4 なら 12 記事で 1 分ほど
df["lead"] = [t.strip().split("。")[0][:30] for t in df["text"]]      # ベースライン: 本文の最初の文の先頭 30 字
for _, r in df.head(5).iterrows():
    print(f"実際: {r['title']}\n生成: {r['generated']}\n先頭: {r['lead']}\n")

実際: 「自分から好きとは言う子じゃなかった」 浅田真央の幼少時代を知る番組ADとは？
生成: 浅田真央、トークで笑顔満開 - フィギュア女王の意外な一面を披露
先頭: 日本テレビ「おしゃれイズム」（27日放送分）では、フィギュア

実際: ボールが顔面を直撃した石井一久＆中村剛也、壮絶エピソード語る
生成: 中村剛也、顔面デッドボールの顛末を語る
先頭: 2日放送、日本テレビ系列「ダウンタウンDX」には、埼玉西武ラ

実際: 【Sports Watch】フジのK-1中継時、アダルトグッズの名前が画面に!?
生成: ベン・エドワーズ、K-1最短記録＆アダルトグッズで話題に
先頭: 7月10日、豪州のキャンベラ・ナショナルコンベンションセンタ

実際: 和田アキ子、スザンヌから斉藤和巳を紹介されるも・・・
生成: 斉藤和巳とスザンヌ、遠距離恋愛実らせ結婚発表 - 和田アキ子も祝福
先頭: プロ野球・ソフトバンクホークスの斉藤和巳と、タレントのスザン

実際: 【Sports Watch】カズぶっちゃけ「息子が不二子ちゃんの服脱がして……」
生成: カズ伝説再び：三浦知良、先輩ROCK YOUで「あしたのジョー」＆「ルパン」愛を語る
先頭: 日本テレビ「心ゆさぶれ！先輩ROCK YOU」（2日放送分）



In [3]:
import numpy as np
from fugashi import Tagger
from rouge_score import rouge_scorer

tagger = Tagger()

class Tok:
    def tokenize(self, text):
        return [w.surface for w in tagger(text)]

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], tokenizer=Tok())
for col, label in [("generated", "LLM の見出し"), ("lead", "本文の先頭 30 字")]:
    scores = [scorer.score(t, g) for t, g in zip(df["title"], df[col])]
    print(f"{label:12} ROUGE-1 F = {np.mean([s['rouge1'].fmeasure for s in scores]):.3f}  ROUGE-L F = {np.mean([s['rougeL'].fmeasure for s in scores]):.3f}  平均 {df[col].str.len().mean():.0f} 字")

LLM の見出し     ROUGE-1 F = 0.332  ROUGE-L F = 0.266  平均 27 字
本文の先頭 30 字   ROUGE-1 F = 0.169  ROUGE-L F = 0.140  平均 30 字


- 見出し生成は、要約の中でも最も圧縮率が高いタスク。何を残すかの選択がそのまま品質になる。
- 実際の見出しは、読者の興味を引くための表現（煽り、伏せ字、疑問形）を含むことが多く、内容の要約とは限らない。ROUGE が低くても、見出しとして悪いとは言えない。
- 12 記事では平均値のばらつきが大きい。手法を比べるなら、数百件以上で評価し、信頼区間を付ける（評価指標と統計的検定の章を参照）。
- 「記事にない情報は入れない」と指示しても、固有名詞や数字を作ってしまうことがある。公開する見出しに使うなら、人の確認が必要。